# 02 — Data Cleaning

Builds on 01_data_loading.ipynb. Filters to London, computes Hi/Ci from TS045, cleans IoD2025 and OpenStreetEV_GLA, and cleans the Zapmap session data (dedup + interval-merge + window-clip so ur_j is mathematically bounded in [0,1]).

## 0. Setup

In [17]:
import pandas as pd
import numpy as np
import os
import re

BASE = "/Users/alexia/Documents/CASA/Dissertation"
print("Base path:", BASE)

Base path: /Users/alexia/Documents/CASA/Dissertation


## 1. Reload raw datasets (independent of 01_data_loading.ipynb)

In [18]:
# TS045 (car/van availability by household)
ts045_path = os.path.join(BASE, "03_data/demand/census2021/TS045_car_van_availability_London_LSOA_2021.csv")
ts045_raw = pd.read_csv(ts045_path, skiprows=7).rename(columns={
    "Area": "area_raw",
    "No cars or vans in household": "cars_0",
    "1 car or van in household": "cars_1",
    "2 cars or vans in household": "cars_2",
    "3 or more cars or vans in household": "cars_3plus",
})
ts045_raw = ts045_raw.dropna(subset=["area_raw"])
for col in ["cars_0", "cars_1", "cars_2", "cars_3plus"]:
    ts045_raw[col] = pd.to_numeric(ts045_raw[col], errors="coerce")
ts045_gor_london = ts045_raw[ts045_raw["area_raw"] == "gor:London"].copy()
ts045_lsoa = ts045_raw[ts045_raw["area_raw"].str.startswith("lsoa2021:", na=False)].copy()
parsed = ts045_lsoa["area_raw"].str.extract(r"lsoa2021:(?P<lsoa_code>\S+)\s*:\s*(?P<lsoa_name>.+)")
ts045_lsoa["lsoa_code"] = parsed["lsoa_code"]
ts045_lsoa["lsoa_name"] = parsed["lsoa_name"]
ts045 = ts045_lsoa[["lsoa_code", "lsoa_name", "cars_0", "cars_1", "cars_2", "cars_3plus"]].reset_index(drop=True)

# IoD2025 File 7
iod_raw = pd.read_csv(os.path.join(BASE, "03_data/demand/IoD2025.csv"))
cols_keep = [
    "LSOA code (2021)", "LSOA name (2021)",
    "Local Authority District code (2024)", "Local Authority District name (2024)",
    "Index of Multiple Deprivation (IMD) Score",
    "Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)",
    "Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs)",
    "Income Score (rate)", "Income Rank (where 1 is most deprived)",
    "Income Decile (where 1 is most deprived 10% of LSOAs)",
]
iod2025 = iod_raw[cols_keep].rename(columns={
    "LSOA code (2021)": "lsoa_code", "LSOA name (2021)": "lsoa_name",
    "Local Authority District code (2024)": "lad_code",
    "Local Authority District name (2024)": "lad_name",
    "Index of Multiple Deprivation (IMD) Score": "imd_score",
    "Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)": "imd_rank",
    "Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs)": "imd_decile",
    "Income Score (rate)": "income_score",
    "Income Rank (where 1 is most deprived)": "income_rank",
    "Income Decile (where 1 is most deprived 10% of LSOAs)": "income_decile",
})

# OpenStreetEV GLA
osev_raw = pd.read_csv(os.path.join(BASE, "03_data/restricted/gla/OpenStreetEV_GLA.csv"))
osev = osev_raw[["country_co","id","external_l","name","address","postal_cod","state",
                  "coordinate","coordina_1","location_c","location_1"]].rename(columns={
    "country_co": "country_code", "id": "location_id", "external_l": "external_uuid",
    "name": "location_name", "postal_cod": "postcode", "state": "borough",
    "coordinate": "latitude", "coordina_1": "longitude",
    "location_c": "location_category", "location_1": "location_type",
})

# Zapmap (join_august2025)
zapmap_raw = pd.read_csv(os.path.join(BASE, "03_data/restricted/join_august2025.csv"), low_memory=False)
cols_keep = ["location_id","evse_id","address","city","state","coordinate","coordina_1",
             "location_c","zapmap_device_uid","power_band","device_max_power",
             "uid","charing_start","status","charging_duration","month","year","day"]
zapmap = zapmap_raw[cols_keep].rename(columns={
    "state": "borough", "coordinate": "latitude", "coordina_1": "longitude",
    "location_c": "location_category", "charing_start": "charging_start", "uid": "session_uid",
})

print("All datasets reloaded.")
print(f"ts045: {ts045.shape}, iod2025: {iod2025.shape}, osev: {osev.shape}, zapmap: {zapmap.shape}")

All datasets reloaded.
ts045: (35672, 6), iod2025: (33755, 10), osev: (23045, 11), zapmap: (215037, 18)


In [19]:
output_dir = os.path.join(BASE, "05_processed")
os.makedirs(output_dir, exist_ok=True)
print("Output directory:", output_dir)

Output directory: /Users/alexia/Documents/CASA/Dissertation/05_processed


## 2. TS045: filter to London + calculate Hi and Ci

In [20]:
london_boroughs = [
    "City of London", "Barking and Dagenham", "Barnet", "Bexley", "Brent",
    "Bromley", "Camden", "Croydon", "Ealing", "Enfield", "Greenwich",
    "Hackney", "Hammersmith and Fulham", "Haringey", "Harrow", "Havering",
    "Hillingdon", "Hounslow", "Islington", "Kensington and Chelsea",
    "Kingston upon Thames", "Lambeth", "Lewisham", "Merton", "Newham",
    "Redbridge", "Richmond upon Thames", "Southwark", "Sutton",
    "Tower Hamlets", "Waltham Forest", "Wandsworth", "Westminster"
]

def is_london(name, boroughs):
    if not isinstance(name, str):
        return False
    return any(re.match(rf"^{re.escape(b)}\s+\d", name) for b in boroughs)

ts045_london = ts045[ts045["lsoa_name"].apply(lambda x: is_london(x, london_boroughs))].reset_index(drop=True)
print("TS045 London rows:", len(ts045_london))

TS045 London rows: 4994


In [21]:
band_cols = ["cars_0", "cars_1", "cars_2", "cars_3plus"]
ts045_london["households"] = ts045_london[band_cols].sum(axis=1)
ts045_london["avg_car_ownership"] = (
    0 * ts045_london["cars_0"] + 1 * ts045_london["cars_1"]
    + 2 * ts045_london["cars_2"] + 3 * ts045_london["cars_3plus"]
) / ts045_london["households"]

census_london = ts045_london[["lsoa_code", "lsoa_name", "households", "avg_car_ownership"] + band_cols].copy()
census_london = census_london.rename(columns={"households": "Hi", "avg_car_ownership": "Ci"})

print("=== Census London (TS045-derived) ===")
print("Shape:", census_london.shape)
print(census_london.head(3).to_string())

gor_total_households = ts045_gor_london[band_cols].sum(axis=1).values[0]
computed_total = census_london["Hi"].sum()
print()
print(f"Computed total households: {computed_total:,.0f}")
print(f"Official gor:London aggregate: {gor_total_households:,.0f}")
print(f"Difference: {(computed_total - gor_total_households) / gor_total_households:.4%}")

output_path = os.path.join(BASE, "05_processed/census_london_clean.csv")
census_london.to_csv(output_path, index=False)
print("\nSaved to:", output_path)

=== Census London (TS045-derived) ===
Shape: (4994, 8)
   lsoa_code            lsoa_name      Hi        Ci  cars_0  cars_1  cars_2  cars_3plus
0  E01000001  City of London 001A   837.0  0.395460     555   243.0    29.0        10.0
1  E01000002  City of London 001B   824.0  0.359223     578   208.0    26.0        12.0
2  E01000003  City of London 001C  1017.0  0.216323     826   169.0    15.0         7.0

Computed total households: 3,423,845
Official gor:London aggregate: 3,423,890
Difference: -0.0013%

Saved to: /Users/alexia/Documents/CASA/Dissertation/05_processed/census_london_clean.csv


## 3. IoD2025: filter to London

In [22]:
london_codes = set(census_london["lsoa_code"])
iod2025_london = iod2025[iod2025["lsoa_code"].isin(london_codes)].reset_index(drop=True)

matched = len(iod2025_london)
expected = len(london_codes)
print(f"Matched: {matched} / Expected: {expected}")
print()
print("income_score stats:")
print(iod2025_london["income_score"].describe())

output_path = os.path.join(BASE, "05_processed/imd_london_clean.csv")
iod2025_london.to_csv(output_path, index=False)
print("\nSaved to:", output_path)

Matched: 4994 / Expected: 4994

income_score stats:
count    4994.000000
mean        0.270942
std         0.147900
min         0.010000
25%         0.152000
50%         0.262000
75%         0.372000
max         0.998000
Name: income_score, dtype: float64

Saved to: /Users/alexia/Documents/CASA/Dissertation/05_processed/imd_london_clean.csv


## 4. Zapmap: datetime parsing, session dedup, duration cleaning

In [23]:
zapmap["charging_start"] = pd.to_datetime(zapmap["charging_start"], errors="coerce")
parse_failures = zapmap["charging_start"].isna().sum()
print(f"charging_start parse failures: {parse_failures}")

# Drop duplicate session records: join_august2025.csv contains a large number of exact
# content-duplicate rows (same evse_id + charging_start + charging_duration repeated many
# times under different session_uid -- session_uid derives from status-change events, not
# a unique-session key, so it is excluded from the dedup key).
rows_raw = len(zapmap)
zapmap_dedup = zapmap.drop_duplicates(subset=["evse_id", "charging_start", "charging_duration"]).copy()
print(f"Rows before dedup: {rows_raw:,}  |  after: {len(zapmap_dedup):,}  "
      f"|  removed: {rows_raw - len(zapmap_dedup):,} ({(1 - len(zapmap_dedup)/rows_raw):.1%})")

rows_before = len(zapmap_dedup)
zapmap_clean = zapmap_dedup[
    (zapmap_dedup["charging_duration"] > 0) &
    (zapmap_dedup["charging_duration"] <= 10080) &
    (zapmap_dedup["charging_start"].notna())
].copy()
print(f"Rows before duration filter: {rows_before:,}  |  after: {len(zapmap_clean):,}")

charging_start parse failures: 0
Rows before dedup: 215,037  |  after: 63,105  |  removed: 151,932 (70.7%)
Rows before duration filter: 63,105  |  after: 63,105


In [24]:
zapmap_clean["hour"] = zapmap_clean["charging_start"].dt.hour
zapmap_clean["dayofweek"] = zapmap_clean["charging_start"].dt.dayofweek

print("=== Zapmap Cleaned ===")
print("Shape:", zapmap_clean.shape)
print("Unique evse_id:", zapmap_clean["evse_id"].nunique())
print("Unique location_id:", zapmap_clean["location_id"].nunique())

output_path = os.path.join(BASE, "05_processed/zapmap_clean.csv")
zapmap_clean.to_csv(output_path, index=False)
print("\nSaved to:", output_path)

=== Zapmap Cleaned ===
Shape: (63105, 20)
Unique evse_id: 10465
Unique location_id: 7582

Saved to: /Users/alexia/Documents/CASA/Dissertation/05_processed/zapmap_clean.csv


## 5. Per-EVSE ur_j: interval-merged, window-clipped

In [25]:
# Merge overlapping session intervals per EVSE (a single long plug-in event can be
# re-logged as several overlapping rows), clipped to the declared 7-day reference window
# (Aug 4 00:00 - Aug 11 00:00, 2025). Clipping guarantees ur_j <= 1.0 by construction.
WINDOW_START = pd.Timestamp("2025-08-04 00:00:00")
WINDOW_END   = WINDOW_START + pd.Timedelta(days=7)

def merged_occupied_minutes(group):
    group = group.sort_values("charging_start")
    total = 0.0
    cur_start = cur_end = None
    for start, dur in zip(group["charging_start"], group["charging_duration"]):
        end = start + pd.Timedelta(minutes=dur)
        if cur_start is None:
            cur_start, cur_end = start, end
        elif start <= cur_end:
            cur_end = max(cur_end, end)
        else:
            cs, ce = max(cur_start, WINDOW_START), min(cur_end, WINDOW_END)
            if ce > cs:
                total += (ce - cs).total_seconds() / 60
            cur_start, cur_end = start, end
    if cur_start is not None:
        cs, ce = max(cur_start, WINDOW_START), min(cur_end, WINDOW_END)
        if ce > cs:
            total += (ce - cs).total_seconds() / 60
    return total

merged_duration = (
    zapmap_clean.groupby("evse_id")
    .apply(merged_occupied_minutes, include_groups=False)
    .reset_index(name="total_duration_min")
)
evse_coords = zapmap_clean.groupby("evse_id")[["location_id", "latitude", "longitude"]].first().reset_index()
evse_ur = merged_duration.merge(evse_coords, on="evse_id", how="left")
evse_ur["ur_j"] = evse_ur["total_duration_min"] / 10080

print("=== Per-EVSE ur_j ===")
print("Unique EVSEs with >=1 session:", len(evse_ur))
print(evse_ur["ur_j"].describe())
print(f"EVSEs with ur_j > 1: {(evse_ur['ur_j'] > 1).sum()} (should be 0)")

output_path = os.path.join(BASE, "05_processed/evse_ur_clean.csv")
evse_ur.to_csv(output_path, index=False)
print("\nSaved to:", output_path)

=== Per-EVSE ur_j ===
Unique EVSEs with >=1 session: 10465
count    10465.000000
mean         0.229637
std          0.276919
min          0.000099
25%          0.000198
50%          0.100794
75%          0.369147
max          0.999504
Name: ur_j, dtype: float64
EVSEs with ur_j > 1: 0 (should be 0)

Saved to: /Users/alexia/Documents/CASA/Dissertation/05_processed/evse_ur_clean.csv


## 6. OpenStreetEV: duplicate + bounding box check

In [26]:
print("=== OpenStreetEV Basic Checks ===")
print(f"Total rows: {len(osev)}")
print(f"Duplicate location_id: {osev.duplicated(subset=['location_id']).sum()}")

outside_bbox = osev[
    (osev["latitude"] < 51.28) | (osev["latitude"] > 51.70) |
    (osev["longitude"] < -0.51) | (osev["longitude"] > 0.33)
]
print(f"Rows outside London bounding box: {len(outside_bbox)}")

osev_clean = osev.drop_duplicates(subset=["location_id"], keep="first")
osev_clean = osev_clean[
    (osev_clean["latitude"] >= 51.28) & (osev_clean["latitude"] <= 51.70) &
    (osev_clean["longitude"] >= -0.51) & (osev_clean["longitude"] <= 0.33)
].reset_index(drop=True)

print()
print("=== OpenStreetEV Cleaned ===")
print("Shape:", osev_clean.shape)
print("location_category value counts:")
print(osev_clean["location_category"].value_counts())
print(f"On-Street locations: {(osev_clean['location_category'] == 'On-Street').sum()}")

output_path = os.path.join(BASE, "05_processed/osev_london_clean.csv")
osev_clean.to_csv(output_path, index=False)
print("\nSaved to:", output_path)

=== OpenStreetEV Basic Checks ===
Total rows: 23045
Duplicate location_id: 30
Rows outside London bounding box: 0

=== OpenStreetEV Cleaned ===
Shape: (23015, 11)
location_category value counts:
location_category
On-Street              21366
Car Park                 406
Supermarket              315
Fuel Forecourt           138
Accommodation            135
Workplace Car Park       129
Leisure                  109
Retail Car Park           96
Restaurant/Pub/Cafe       76
Health Services           54
Other                     49
Education                 41
Travel Interchange        35
Public Services           33
Dealership                25
Motorway Services          7
Park and Ride              1
Name: count, dtype: int64
On-Street locations: 21366

Saved to: /Users/alexia/Documents/CASA/Dissertation/05_processed/osev_london_clean.csv


## 7. Pipeline summary

In [27]:
summary_data = [
    ("LSOAs loaded (Greater London)", len(census_london)),
    ("Household total (Hi sum, validated vs gor:London)", int(census_london["Hi"].sum())),
    ("Total EVSE locations (OpenStreetEV_GLA, all categories)", len(osev_clean)),
    ("On-street EVSE locations (OpenStreetEV_GLA)", int((osev_clean["location_category"] == "On-Street").sum())),
    ("Session records (join_august2025, post-clean)", len(zapmap_clean)),
    ("Unique EVSEs with >=1 session (ur_j computed)", len(evse_ur)),
    ("eⱼ / Sᵢᵉᶠᶠ / Uᵢ / has_evse_i (LSOA-level)", "Pending — needs LSOA boundary + on-street filter (03_EDA)"),
]
pipeline_summary = pd.DataFrame(summary_data, columns=["Item", "Count"])
print(pipeline_summary.to_string(index=False))

output_path = os.path.join(BASE, "05_processed/pipeline_summary.csv")
pipeline_summary.to_csv(output_path, index=False)
print("\nSaved to:", output_path)

                                                   Item                                                     Count
                          LSOAs loaded (Greater London)                                                      4994
      Household total (Hi sum, validated vs gor:London)                                                   3423845
Total EVSE locations (OpenStreetEV_GLA, all categories)                                                     23015
            On-street EVSE locations (OpenStreetEV_GLA)                                                     21366
          Session records (join_august2025, post-clean)                                                     63105
          Unique EVSEs with >=1 session (ur_j computed)                                                     10465
              eⱼ / Sᵢᵉᶠᶠ / Uᵢ / has_evse_i (LSOA-level) Pending — needs LSOA boundary + on-street filter (03_EDA)

Saved to: /Users/alexia/Documents/CASA/Dissertation/05_processed/pipeline_summary.csv
